In [26]:
from lightning.pytorch.callbacks import ModelCheckpoint
from loaders.Sndataloader2 import SNpart_Dataset
from model.GNN_inf_seg import Lightning_GNN
import torch_geometric as tg
import numpy as np
import lightning as pl
import datetime
import yaml
import os
import wandb
import torch
import pyvista as pv

In [241]:
with open('configs/config_SNpart.yml', 'r') as f:
    config = yaml.safe_load(f)

config['batch_size'] = 10

# Data setup
dataset_test = SNpart_Dataset(root=config['root'],
                                 split='test')

test_loader = tg.loader.DataLoader(dataset_test,
                                  batch_size=config['batch_size'],
                                  num_workers=2,
                                  shuffle=True)

# Model setup
GNN_model = Lightning_GNN(config=config)
GNN_model.load_state_dict(torch.load('/home/lars/models/2024-08-22_20.41.44SN_part_blocks_base_wskip/epoch=199-train_loss=0.12.ckpt')['state_dict'])
GNN_model.to('cpu')
a = 1

In [242]:
with torch.no_grad():
    GNN_model.eval()
    sample = next(iter(test_loader))
    out_pc = GNN_model(sample)

prediction = torch.argmax(out_pc, dim=1)

In [243]:
# extract sample with lowest accuracy   
accr_list = [] 

for i in range(config['batch_size']):
    accr = torch.sum(prediction[sample.batch == i] == sample.y[sample.batch == i]).item() / len(sample.y[sample.batch == i])
    accr_list.append((accr, i))

lowest_accuracy_indices = [x[1] for x in sorted(accr_list, key=lambda x: x[0])[:3]]
print(lowest_accuracy_indices)

[7, 5, 2]


In [245]:
batch_idx = 5

In [246]:
pos = sample.pos[sample.batch == batch_idx].numpy()
pred = prediction[sample.batch == batch_idx].numpy()
label = sample.y[sample.batch == batch_idx].numpy()

In [236]:
(label == 49).sum()

4

In [255]:
# Create a PyVista plotter
plotter = pv.Plotter()
point_cloud_pv = pv.PolyData(pos)
point_cloud_pv['labels'] = label
plotter.add_points(point_cloud_pv, scalars='labels', cmap='viridis', render_points_as_spheres=True)
plotter.camera_position = [
    (2.0, 0.4, -2.0),   # Replace with extracted camera position
    (0.0, 0.0, 0.0),   # Replace with extracted focal point
    (-0.6, -0.5, 0.76)    # Replace with extracted view-up vector
]
#plotter.view_vector([0.36806499, 0.17783007,  -0.91263609])
plotter.show()

Widget(value='<iframe src="http://localhost:44033/index.html?ui=P_0x797415748da0_116&reconnect=auto" class="py…

In [254]:
camera_position = np.array(plotter.camera_position[0])  # Camera position
focal_point = np.array(plotter.camera_position[1])      # Focal point

# Compute the view vector (direction vector)
view_vector = focal_point - camera_position

# Normalize the view vector to get the direction
view_vector = view_vector / np.linalg.norm(view_vector)

print("Current view vector:", view_vector)
print("Current camera position:", camera_position)

Current view vector: [-0.72559271 -0.14964974 -0.67165481]
Current camera position: [2.05228609 0.42327339 1.89972668]


In [256]:
# Create a new PyVista plotter for the output point cloud
plotter_output = pv.Plotter()
point_cloud_pv['output_labels'] = pred
plotter_output.add_points(point_cloud_pv, scalars='output_labels', cmap='viridis', render_points_as_spheres=True)
plotter_output.camera_position = [
    (2.0, 0.4, -2.0),   # Replace with extracted camera position
    (0.0, 0.0, 0.0),   # Replace with extracted focal point
    (-0.6, -0.5, 0.76)    # Replace with extracted view-up vector
]
plotter_output.show()

Widget(value='<iframe src="http://localhost:44033/index.html?ui=P_0x7974157486e0_117&reconnect=auto" class="py…